<a href="https://colab.research.google.com/github/R4j4n/Autonotes/blob/master/Ollama_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,199 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,639 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jamm

In [24]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [8]:
!ollama pull phi4

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling fd7b6731c33c...   0% ▕▏    0 B/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   0% ▕▏    0 B/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   0% ▕▏    0 B/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   0% ▕▏ 5.5 MB/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   0% ▕▏  12 MB/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   0% ▕▏  31 MB/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   1% ▕▏  51 MB/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   1% ▕▏  65 MB/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   1% ▕▏  90 MB/9.1 GB                  pulling manifest 
pulling fd7b6731c33c...   1% ▕▏ 132 MB/9.1 GB                  pulling manifest 
pulling fd7b6731c33c

In [4]:
!pip install langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1


In [28]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "run", "phi4"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [29]:
# !pip install chromadb
# !pip install llama-index
!pip install -q llama-index-llms-ollama
!pip install -q llama-index-vector-stores-chroma

In [12]:
import chromadb
from dotenv import load_dotenv
from llama_index.llms.ollama import  Ollama
from llama_index.core import Settings
from IPython.display import Markdown, display
from llama_index.core import StorageContext
from IPython.display import HTML
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import ListIndex, SimpleDirectoryReader, VectorStoreIndex

In [16]:
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings


# Set up Ollama
llm = Ollama(model="phi4")
Settings.llm = llm


In [19]:
import time
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core.node_parser import SentenceSplitter

class Ingest:

    def __init__(self,collection_name="test", save_dir = "ingested", overwrite = False):

        self.overwrite = overwrite
        self.chroma_client = chromadb.PersistentClient(path=f"{save_dir}")
        self.collection_name = collection_name

        self.embed_model =  HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5",device="cuda")


        self.collection = self.load_or_create_collection()

        self.save_dir = save_dir


    def load_or_create_collection(self):
        existing_collections = self.chroma_client.list_collections()
        print(f"Existing Collections:{existing_collections}")

        if (self.collection_name in existing_collections):

            if self.overwrite:
                self.chroma_client.delete_collection(self.collection_name)
                chroma_collection = self.chroma_client.create_collection(self.collection_name)
                print(f"Created new collection '{self.collection_name}'.")
            else:
                chroma_collection = self.chroma_client.get_collection(self.collection_name)
                print(f"Using existing collection '{self.collection_name}'.")
        else:
            chroma_collection = self.chroma_client.create_collection(self.collection_name)
            print(f"Created new collection '{self.collection_name}'.")

        return chroma_collection

    def ingest(self,documents):
        start_time = time.time()
        vector_store = ChromaVectorStore(chroma_collection=self.collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context, embed_model=self.embed_model,
        transformations=[SentenceSplitter(chunk_size=256, chunk_overlap=20)]
        )
        end_time = time.time()

        print(f"Time taken for indexing: {(end_time-start_time)/60} seconds.")
        return index

In [20]:
ingester = Ingest(collection_name="Rajan",overwrite=True)
documents = SimpleDirectoryReader(input_files=["/content/data.pdf"]).load_data()

index = ingester.ingest(documents=documents)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Existing Collections:[]
Created new collection 'Rajan'.
Time taken for indexing: 0.04363783597946167 seconds.


In [30]:

from llama_index.core import (
    get_response_synthesizer,
    QueryBundle
)
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.prompts import PromptTemplate
from llama_index.core.postprocessor import SentenceTransformerRerank, LLMRerank
from llama_index.core.query_engine import RetrieverQueryEngine

class RAG:

    def __init__(self, index ,top_k : int = 5, use_reranker=False, reranker_top_n = 5):
        llm = Ollama(model="phi4", temperature=0)
        Settings.llm =  llm
        Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
        Settings.chunk_overlap = 0
        Settings.chunk_size = 256

        Settings.text_splitter = SentenceSplitter(chunk_size=256, chunk_overlap=20)

        self.retriever = VectorIndexRetriever(index=index, similarity_top_k=top_k)
        self.response_synthesizer = get_response_synthesizer()

        self.use_reranker = use_reranker
        self.reranker_top_n = reranker_top_n

        template = """
           <s> [Instructions] You are a AeroSports Trampoline Park's customer support assistant. Answer the question based only on the following context.
        If you don't know the answer, then reply, sorry i dont have any idea about the {question}. [/Instructions] </s>
        [Instructions] Question: {question}
        Context: {context}
        Answer: [/Instructions]
        """
        self.prompt_tmpl = PromptTemplate(
            template=template,
            template_var_mappings={"query_str": "question", "context_str": "context"}
        )


    def __call__(self,query_str):

        query_bundle = QueryBundle(query_str)

        retrieved_nodes = self.retriever.retrieve(query_bundle)

        # Rerank nodes if reranker is enabled
        if self.use_reranker:
            reranker = SentenceTransformerRerank(
                model="cross-encoder/ms-marco-MiniLM-L-2-v2",
                top_n=self.reranker_top_n
            )

            # reranker = LLMRerank(llm=llm,choice_batch_size=5,top_n=reranker_top_n,)
            retrieved_nodes = reranker.postprocess_nodes(retrieved_nodes, query_bundle)
        else:
            reranker = None

        # Assemble the query engine
        query_engine = RetrieverQueryEngine(
            retriever=self.retriever,
            response_synthesizer=self.response_synthesizer,
            node_postprocessors=[reranker] if reranker else []
        )

        # Update query engine with custom prompt template
        query_engine.update_prompts(
            {"response_synthesizer:text_qa_template": self.prompt_tmpl}
        )

        # Get synthesized answer
        answer = query_engine.query(query_bundle)

        return retrieved_nodes, answer


In [31]:
rag = RAG(index=index)
from IPython.display import Markdown

In [57]:
x, y = rag(query_str="My birthday starts at 1 PM but my guests will come early, can I use the party room 30 minutes early?")
display(Markdown(y.response))

Based on the provided context, there isn't specific information regarding using the party room 30 minutes early before your scheduled start time at 1 PM. The closest relevant detail is that guests should arrive 15 minutes early unless the party is scheduled at the park's opening time. However, this does not explicitly cover using the party room earlier than planned.

If you need to use the party room 30 minutes early, I recommend contacting AeroSports directly for clarification or special arrangements. They may be able to accommodate your request based on availability and specific circumstances. 

Sorry, I don't have any idea about whether you can use the party room 30 minutes early without further confirmation from AeroSports.

In [62]:
x, y = rag(query_str="What is the cancellation policy for parties?")
display(Markdown(y.response))

The cancellation policy for parties at AeroSports Trampoline Park is as follows:

- Parties can be canceled up to 2 weeks before the event date to receive a refund in the form of store credit.
- If a party is canceled within 2 weeks of the event date, no refunds will be offered. However, the party can be rescheduled for up to 1 year from the original date.

If you have any more questions or need further assistance, feel free to ask!